In [40]:
# %%
# ============================================================
# 🧠 FULL SUMMARIZATION KD PIPELINE
# Universal Teacher + Sequence-Level KD
# ============================================================

from typing import TypedDict, Any, List,Optional
import torch
import unicodedata
import os
import math

from datasets import load_dataset

import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM,AutoConfig,AutoModelForCausalLM
from torch.optim import AdamW
from langgraph.graph import StateGraph, END
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

import evaluate


In [41]:

# %%
class SLMGenState(TypedDict):

    task: str
    user_quary:Optional[str]
    dataset_name: str
    dataset: Any
    dataset_config: Optional[dict]

    detected_input_field: Optional[str]
    detected_target_field: Optional[str]

    distillation_plan: dict

    clean_dataset: Any
    processed_dataset: Any

    tokenized_dataset: Any
    student_tokenizer: Any

    teacher_type: str
    teacher_model: str
    teacher_model_instance: Any
    teacher_tokenizer: Any

    teacher_logits: Any
    teacher_outputs: List[str]

    student_model: str
    student_model_instance: Any

    evaluation_plan: Optional[dict]

    evaluation_metrics: dict
    student_outputs: list

    save_path: str


In [42]:

# ============================================================
# 📚 UNIVERSAL NLP DATASET REGISTRY
# ============================================================

AVAILABLE_DATASETS = {

    # ========================================================
    # 📝 Summarization
    # ========================================================
    "summarization": [
        {
            "name": "cnn_dailymail",
            "hf_id": "cnn_dailymail",
            "config": "3.0.0",
            "input_field": "article",
            "target_field": "highlights",
            "metric": "rouge"
        },
        {
            "name": "xsum",
            "hf_id": "xsum",
            "config": None,
            "input_field": "document",
            "target_field": "summary",
            "metric": "rouge"
        },
        {
            "name": "samsum",
            "hf_id": "samsum",
            "config": None,
            "input_field": "dialogue",
            "target_field": "summary",
            "metric": "rouge"
        }
    ],

    # ========================================================
    # 🌍 Translation
    # ========================================================
    "translation": [
        {
            "name": "samanantar_en_ml",
            "hf_id": "ai4bharat/samanantar",
            "config": "ml",
            "input_field": "translation",
            "source_lang": "en",
            "target_lang": "ml",
            "metric": "bleu"
        },
        {
            "name": "wmt14_en_de",
            "hf_id": "wmt14",
            "config": "de-en",
            "input_field": "translation",
            "source_lang": "en",
            "target_lang": "de",
            "metric": "bleu"
        },
        {
            "name": "opus_books_en_fr",
            "hf_id": "opus_books",
            "config": "en-fr",
            "input_field": "translation",
            "source_lang": "en",
            "target_lang": "fr",
            "metric": "bleu"
        }
    ],

    # ========================================================
    # ❓ Question Answering
    # ========================================================
    "question_answering": [
        {
            "name": "squad_v1",
            "hf_id": "squad",
            "config": None,
            "context_field": "context",
            "question_field": "question",
            "answer_field": "answers",
            "metric": "squad"
        },
        {
            "name": "squad_v2",
            "hf_id": "squad_v2",
            "config": None,
            "context_field": "context",
            "question_field": "question",
            "answer_field": "answers",
            "metric": "squad_v2"
        }
    ],

    # ========================================================
    # 🏷️ Text Classification
    # ========================================================
    "text_classification": [
        {
            "name": "ag_news",
            "hf_id": "ag_news",
            "config": None,
            "input_field": "text",
            "target_field": "label",
            "metric": "accuracy"
        },
        {
            "name": "imdb",
            "hf_id": "imdb",
            "config": None,
            "input_field": "text",
            "target_field": "label",
            "metric": "accuracy"
        },
        {
            "name": "sst2",
            "hf_id": "glue",
            "config": "sst2",
            "input_field": "sentence",
            "target_field": "label",
            "metric": "accuracy"
        }
    ],

    # ========================================================
    # 🧠 Natural Language Inference
    # ========================================================
    "nli": [
        {
            "name": "mnli",
            "hf_id": "glue",
            "config": "mnli",
            "premise_field": "premise",
            "hypothesis_field": "hypothesis",
            "target_field": "label",
            "metric": "accuracy"
        }
    ],

    # ========================================================
    # 💬 Dialogue / Instruction Following
    # ========================================================
    "instruction_tuning": [
        {
            "name": "alpaca",
            "hf_id": "tatsu-lab/alpaca",
            "config": None,
            "input_field": "instruction",
            "output_field": "output",
            "metric": "bleu"
        },
        {
        "name": "dolly_15k",
        "hf_id": "databricks/databricks-dolly-15k",
        "config": None,
        "instruction_field": "instruction",
        "context_field": "context",
        "output_field": "response",
        "category_field": "category",
        "metric": "bleu"
    }
    ],

    # ========================================================
    # 🧾 Paraphrasing
    # ========================================================
    "paraphrase": [
        {
            "name": "qqp",
            "hf_id": "glue",
            "config": "qqp",
            "question1_field": "question1",
            "question2_field": "question2",
            "target_field": "label",
            "metric": "accuracy"
        }
    ]
}


In [43]:


from huggingface_hub import list_datasets
from itertools import islice



dataset_selector_llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0.3,
)


DATASET_SELECTOR_PROMPT = PromptTemplate(
    input_variables=["task","user_quary", "dataset_options"],
    template="""
You are an NLP dataset selector.

Task: {task}

user task:{user_quary}

Available datasets:
{dataset_options}

Output ONLY dataset name.
"""
)


# ============================================================
# 🧠 DATASET SELECTOR AGENT (Registry + HF Fallback)
# ============================================================

def dataset_selector_agent(state: SLMGenState):

    task = state["task"]
    user_quary = state["user_quary"]
     
    # ========================================================
    # 1️⃣ If Task Exists in Registry
    # ========================================================
    if task in AVAILABLE_DATASETS:

        dataset_names = [
            d["name"] for d in AVAILABLE_DATASETS[task]
        ]

        dataset_options = "\n".join(dataset_names)

        chain = (
            DATASET_SELECTOR_PROMPT
            | dataset_selector_llm
            | StrOutputParser()
        )

        dataset_name = chain.invoke({
            "task": task,
            "user_quary": user_quary,
            "dataset_options": dataset_options
        }).strip()

        print(f"✅ Selected from registry: {dataset_name}")
        return {**state, "dataset_name": dataset_name}


    # ========================================================
    # 2️⃣ If Not in Registry → Search Hugging Face Hub
    # ========================================================
    print("⚠️ Task not in registry. Searching Hugging Face Hub...")

    datasets = list(islice(list_datasets(search=task), 50))

    # Filter private/gated
    datasets = [
        d for d in datasets
        if not d.private and not d.gated
    ]

    # Rank by likes
    ranked = sorted(
        datasets,
        key=lambda x: x.likes if x.likes else 0,
        reverse=True
    )

    if not ranked:
        raise ValueError("No datasets found on Hugging Face Hub.")

    # Take top 5 candidates
    top_candidates = ranked[:5]
    dataset_options = "\n".join([d.id for d in top_candidates])

    chain = (
        DATASET_SELECTOR_PROMPT
        | dataset_selector_llm
        | StrOutputParser()
    )

    dataset_name = chain.invoke({
        "task": task,
        "dataset_options": dataset_options
    }).strip()

    print(f"🌍 Selected from HuggingFace Hub: {dataset_name}")

    return {**state, "dataset_name": dataset_name}


In [44]:

from datasets import load_dataset

# ============================================================
# 📦 DATASET LOADER AGENT
# ============================================================

def dataset_loader_agent(state: SLMGenState):

    task = state["task"]
    dataset_name = state["dataset_name"]

    print(f"📦 Loading dataset: {dataset_name}")

    # ========================================================
    # 1️⃣ Check if dataset exists in registry
    # ========================================================
    if task in AVAILABLE_DATASETS:

        for dataset in AVAILABLE_DATASETS[task]:
            if dataset["name"] == dataset_name:

                hf_id = dataset["hf_id"]
                config = dataset.get("config")

                dataset_obj = load_dataset(
                    hf_id,
                    config
                )

                # ⭐ LIMIT DATASET SIZE
                if "train" in dataset_obj:
                    dataset_obj["train"] = dataset_obj["train"].select(range(200))

                if "validation" in dataset_obj:
                    dataset_obj["validation"] = dataset_obj["validation"].select(range(20))

                if "test" in dataset_obj:
                    dataset_obj["test"] = dataset_obj["test"].select(range(20))

                print("✅ Loaded from registry configuration")
                print("📊 Dataset limited to 1000 train samples")

                return {
                    **state,
                    "dataset": dataset_obj,
                    "dataset_config": dataset
                }

    # ========================================================
    # 2️⃣ Fallback: Load directly from Hugging Face Hub
    # ========================================================
    print("🌍 Loading directly from Hugging Face Hub...")

    dataset_obj = load_dataset(dataset_name)

    # ⭐ LIMIT DATASET SIZE
    if "train" in dataset_obj:
        dataset_obj["train"] = dataset_obj["train"].select(range(100))

    if "validation" in dataset_obj:
        dataset_obj["validation"] = dataset_obj["validation"].select(range(20))

    if "test" in dataset_obj:
        dataset_obj["test"] = dataset_obj["test"].select(range(20))

    print("📊 Dataset limited to 1000 train samples")

    return {
        **state,
        "dataset": dataset_obj,
        "dataset_config": None
    }


In [45]:

# --------------------------------------------------------
# Planner for knowledge distillation strategy
# --------------------------------------------------------
import json

planner_llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

PLANNING_PROMPT = PromptTemplate(
    input_variables=[
        "task",
        "teacher_type",
        "teacher_model",
        "student_model"
    ],
    template="""
You are an expert AI distillation planner.

Task: {task}

Teacher type: {teacher_type}
Teacher model: {teacher_model}

Student model: {student_model}

Decide the best knowledge distillation strategy.

Rules:

1. If teacher exposes logits (HF model) → prefer logit_kd
2. If teacher is API model → use sequence_kd

Return JSON ONLY:

{{
 "kd_strategy": "...",
 "requires_teacher_logits": true/false,
 "tokenization": "student_only | shared",
 "loss_function": "...",
 "temperature": number,
 "alpha": number,
 "max_input_length": number,
 "max_output_length": number
}}

Do not explain.
Return JSON only.
"""
)

def distillation_planner_agent(state: SLMGenState):

    chain = (
        PLANNING_PROMPT
        | planner_llm
        | StrOutputParser()
    )

    response = chain.invoke({
        "task": state["task"],
        "teacher_type": state["teacher_type"],
        "teacher_model": state["teacher_model"],
        "student_model": state["student_model"]
    })

    try:
        plan = json.loads(response)
    except:
        raise ValueError("Planner returned invalid JSON")

    print("\n🧠 Distillation Plan")
    print(plan)

    return {
        **state,
        "distillation_plan": plan
    }



In [46]:
# --------------------------------------------------------
# 🧠 LLM-BASED FIELD DETECTION AGEN
#  --------------------------------------------------------


field_detector_llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)


FIELD_DETECTION_PROMPT = PromptTemplate(
    input_variables=["task", "keys", "sample"],
    template="""
You are an NLP dataset analyzer.

Task: {task}

Dataset keys:
{keys}

Sample example:
{sample}

Return ONLY in this JSON format:
{{
  "input_field": "...",
  "target_field": "..."
}}

the input_field is the field that contains the text input (e.g. article, dialogue, context, question etc.)
the target_field is the field that contains the text output (e.g. summary, answer, response etc.)

Do not explain.
Do not add text.
Return only JSON.
"""
)


import json

def llm_field_detection_agent(state: SLMGenState):

    dataset = state["dataset"]
    task = state["task"]

    sample = dataset["train"][0]
    keys = list(sample.keys())

    chain = (
        FIELD_DETECTION_PROMPT
        | field_detector_llm
        | StrOutputParser()
    )

    response = chain.invoke({
        "task": task,
        "keys": keys,
        "sample": sample
    })

    try:
        fields = json.loads(response)
    except:
        raise ValueError("LLM returned invalid JSON")

    input_field = fields["input_field"]
    target_field = fields["target_field"]

    print("🔍 LLM Detected:")
    print("Input:", input_field)
    print("Target:", target_field)

    return {
        **state,
        "detected_input_field": input_field,
        "detected_target_field": target_field
    }


In [47]:


# ============================================================
# 🧹 TEXT CLEANING FUNCTION
# ============================================================

from langgraph.func import task
from matplotlib.style import context


def clean_text(text: str):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    return text.strip()


# ============================================================
# 🧠 PREPROCESSING AGENT (SEQUENCE KD READY)
# ============================================================

def preprocessing_agent(state: SLMGenState):

    dataset = state["dataset"]
    task = state["task"]

    input_field = state.get("detected_input_field")
    target_field = state.get("detected_target_field")

    print(f"🧹 Preprocessing for task: {task}")
    print(f"Input field: {input_field}")
    print(f"Target field: {target_field}")

    # --------------------------------------------------------
    # 1️⃣ Instruction Tuning Special Case
    # --------------------------------------------------------
    # --------------------------------------------------------

    if task == "instruction_tuning":

        def preprocess(example):

            instruction = clean_text(example.get("instruction", ""))
            context = clean_text(example.get("input", ""))
            response = clean_text(example.get("output", ""))

            if context:
                input_text = f"Instruction: {instruction}\nInput: {context}\nAnswer:"
            else:
                input_text = f"Instruction: {instruction}\nAnswer:"

            return {
                "input_text": input_text,
                "target_text": response
         }

    # --------------------------------------------------------
    # 2️⃣ Translation Special Case (Nested Dict)
    # --------------------------------------------------------
    elif task == "translation":

        def preprocess(example):

            translation = example[input_field]

            # If detected as tuple like ("translation", "en")
            if isinstance(input_field, tuple):
                source_lang = input_field[1]
                target_lang = target_field[1]

                input_text = clean_text(translation[source_lang])
                target_text = clean_text(translation[target_lang])
            else:
                # fallback
                input_text = clean_text(example[input_field])
                target_text = clean_text(example[target_field])

            return {
                "input_text": input_text,
                "target_text": target_text
            }
        
    elif task == "question_answering":

        def preprocess(example):

            context = clean_text(example["context"])
            question = clean_text(example["question"])

            answer = ""
            if "answers" in example and len(example["answers"]["text"]) > 0:
                answer = example["answers"]["text"][0]

            input_text = f"Context: {context}\nQuestion: {question}\nAnswer:"

            return {
                "input_text": input_text,
                "target_text": clean_text(answer)
            }

    # --------------------------------------------------------
    # 3️⃣ Generic Case (Summarization, Classification, QA)
    # --------------------------------------------------------
    else:

        def preprocess(example):

            input_text = clean_text(example[input_field])
            target_text = clean_text(example[target_field])

            return {
                "input_text": input_text,
                "target_text": target_text
            }

    # --------------------------------------------------------
    # 4️⃣ Apply Mapping
    # --------------------------------------------------------
    processed_dataset = dataset.map(
        preprocess,
        remove_columns=dataset["train"].column_names
    )

    print("✅ Preprocessing completed successfully")

    return {
        **state,
        "processed_dataset": processed_dataset
    }


In [48]:


# ============================================================
# 🔤 TOKENIZER AGENT (PLANNER-AWARE)
# ============================================================

def tokenizer_agent(state: SLMGenState):

    plan = state["distillation_plan"]
    dataset = state["processed_dataset"]
    student_model = state["student_model"]

    kd_strategy = plan["kd_strategy"]
    max_input_length = plan["max_input_length"]

    print("\n🔤 Tokenizer Agent")
    print("KD Strategy:", kd_strategy)

    tokenizer = AutoTokenizer.from_pretrained(student_model)

    # --------------------------------------------------------
    # LOGIT KD
    # --------------------------------------------------------
    if kd_strategy in ["logit_kd", "hybrid_kd"]:

        max_output_length = plan["max_output_length"]

        def tokenize(example):

            model_inputs = tokenizer(
                example["input_text"],
                max_length=max_input_length,
                truncation=True,
                padding="max_length"
            )

            labels = tokenizer(
                example["target_text"],
                max_length=max_output_length,
                truncation=True,
                padding="max_length"
            )

            model_inputs["labels"] = labels["input_ids"]

            return model_inputs

    # --------------------------------------------------------
    # SEQUENCE KD
    # --------------------------------------------------------
    elif kd_strategy == "sequence_kd":

        max_output_length = plan["max_output_length"]

        def tokenize(example):

            model_inputs = tokenizer(
                example["input_text"],
                max_length=max_input_length,
                truncation=True,
                padding="max_length"
            )

            labels = tokenizer(
                example["target_text"],
                max_length=max_output_length,
                truncation=True,
                padding="max_length"
            )

            model_inputs["labels"] = labels["input_ids"]

            return model_inputs

    else:
        raise ValueError(f"Unsupported KD strategy: {kd_strategy}")

    # --------------------------------------------------------
    # Apply Tokenization
    # --------------------------------------------------------
    tokenized_dataset = dataset.map(
        tokenize,
        remove_columns=dataset["train"].column_names
    )

    print("✅ Tokenization complete")

    return {
        **state,
        "tokenized_dataset": tokenized_dataset,
        "student_tokenizer": tokenizer
    }



In [49]:
# ============================================================
# 🧠 TEACHER AGENT (LOAD MODEL ONLY)
# ============================================================

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from openai import OpenAI
from groq import Groq
import torch
import os
from dotenv import load_dotenv

load_dotenv()

'''def teacher_agent(state: SLMGenState):


    teacher_type = state["teacher_type"]
    teacher_model = state["teacher_model"]

    print("\n🧠 Teacher Agent")
    print("Teacher type:", teacher_type)
    print("Teacher model:", teacher_model)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # =====================================================
    # HF TEACHER
    # =====================================================

    if teacher_type == "hf":

        model = AutoModelForSeq2SeqLM.from_pretrained(
            teacher_model
            ).to(device)

        tokenizer = AutoTokenizer.from_pretrained(teacher_model)

        model.eval()

        return {
            **state,
            "teacher_model_instance": model,
            "teacher_tokenizer": tokenizer
        }

    # =====================================================
    # OPENAI TEACHER
    # =====================================================

    elif teacher_type == "openai":
        load_dotenv()
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        return {
            **state,
            "teacher_model_instance": client
        }

    # =====================================================
    # GROQ TEACHER
    # =====================================================

    elif teacher_type == "groq":

        client = Groq(api_key=os.getenv("GROQ_API_KEY"))

        return {
            **state,
            "teacher_model_instance": client
        }
    #=====================================================
    # OLLAMA
    #=====================================================
    
    elif teacher_type == "ollama":

        ollama_model = ChatOllama(
            model=teacher_model,
            temperature=0
        )

        return {
            **state,
            "teacher_model_instance": ollama_model
        }


    else:
        raise ValueError("Unsupported teacher type")'''

'def teacher_agent(state: SLMGenState):\n\n\n    teacher_type = state["teacher_type"]\n    teacher_model = state["teacher_model"]\n\n    print("\n🧠 Teacher Agent")\n    print("Teacher type:", teacher_type)\n    print("Teacher model:", teacher_model)\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n\n    # =====================================================\n    # HF TEACHER\n    # =====================================================\n\n    if teacher_type == "hf":\n\n        model = AutoModelForSeq2SeqLM.from_pretrained(\n            teacher_model\n            ).to(device)\n\n        tokenizer = AutoTokenizer.from_pretrained(teacher_model)\n\n        model.eval()\n\n        return {\n            **state,\n            "teacher_model_instance": model,\n            "teacher_tokenizer": tokenizer\n        }\n\n    # =====================================================\n    # OPENAI TEACHER\n    # =====================================================\n\n    elif teacher_t

In [50]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoTokenizer
)

def teacher_agent(state: SLMGenState):

    teacher_type = state["teacher_type"]
    teacher_model = state["teacher_model"]
    task = state["task"]

    print("\n🧠 Teacher Agent")
    print("Teacher type:", teacher_type)
    print("Teacher model:", teacher_model)
    print("Task:", task)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # =====================================================
    # HF TEACHER
    # =====================================================

    if teacher_type == "hf":

        tokenizer = AutoTokenizer.from_pretrained(teacher_model)

        if task in ["text_classification", "nli", "paraphrase"]:

            model = AutoModelForSequenceClassification.from_pretrained(
                teacher_model
            ).to(device)

        else:

            model = AutoModelForSeq2SeqLM.from_pretrained(
                teacher_model,
                torch_dtype=torch.float16
            ).to(device)

        model.eval()

        return {
            **state,
            "teacher_model_instance": model,
            "teacher_tokenizer": tokenizer
        }

    # =====================================================
    # OPENAI
    # =====================================================

    elif teacher_type == "openai":

        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        return {
            **state,
            "teacher_model_instance": client
        }

    # =====================================================
    # GROQ
    # =====================================================

    elif teacher_type == "groq":

        client = Groq(api_key=os.getenv("GROQ_API_KEY"))

        return {
            **state,
            "teacher_model_instance": client
        }

    # =====================================================
    # OLLAMA
    # =====================================================

    elif teacher_type == "ollama":

        ollama_model = ChatOllama(
            model=teacher_model,
            temperature=0
        )

        return {
            **state,
            "teacher_model_instance": ollama_model
        }

    else:
        raise ValueError("Unsupported teacher type")

In [51]:
from tqdm import tqdm
import torch
import time


def teacher_generation_agent(state: SLMGenState):

    teacher_type = state["teacher_type"]
    teacher_model = state["teacher_model"]
    teacher = state["teacher_model_instance"]
    dataset = state["processed_dataset"]

    teacher_outputs = []

    print("\n🧠 Generating teacher outputs")

    max_new_tokens = state.get("teacher_max_new_tokens", 128)
    temperature = state.get("teacher_temperature", 0)

    # ==========================================
    # HF TEACHER
    # ==========================================

    if teacher_type == "hf":

        tokenizer = state["teacher_tokenizer"]
        device = "cuda" if torch.cuda.is_available() else "cpu"

        teacher.to(device)
        teacher.eval()

        batch_size = 4
        inputs = [ex["input_text"] for ex in dataset["train"]]

        for i in tqdm(range(0, len(inputs), batch_size)):

            batch = inputs[i:i+batch_size]

            encoded = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(device)

            with torch.no_grad():
                outputs = teacher.generate(
                    **encoded,
                    max_new_tokens=max_new_tokens,
                    do_sample=False
                )

            decoded = tokenizer.batch_decode(
                outputs,
                skip_special_tokens=True
            )

            teacher_outputs.extend(decoded)

    # ==========================================
    # OLLAMA TEACHER
    # ==========================================

    elif teacher_type == "ollama":

        for example in tqdm(dataset["train"]):

            prompt = f"""
You are a helpful AI teacher.

Summarize the following text:

{example["input_text"]}
"""

            try:

                response = teacher.invoke(prompt)

                text = response.content if hasattr(response, "content") else str(response)

                if text is None or text.strip() == "":
                    text = example["target_text"]

                teacher_outputs.append(text)

            except Exception as e:

                print(f"⚠️ Ollama error: {e}")
                teacher_outputs.append(example["target_text"])

    # ==========================================
    # OPENAI / GROQ TEACHER
    # ==========================================

    elif teacher_type in ["openai", "groq"]:

        for example in tqdm(dataset["train"]):

            prompt = f"""
                You are a helpful AI teacher.

                Summarize the following text:

                {example["input_text"]}
            """

            for attempt in range(3):

                try:

                    response = teacher.chat.completions.create(
                        model=teacher_model,
                        messages=[
                            {"role": "user", "content": prompt}
                        ],
                        temperature=temperature
                    )

                    text = response.choices[0].message.content

                    if text is None or text.strip() == "":
                        text = example["target_text"]

                    teacher_outputs.append(text)
                    break

                except Exception as e:

                    print(f"⚠️ API error: {e}")
                    time.sleep(2)

                    if attempt == 2:
                        teacher_outputs.append(example["target_text"])

    else:
        raise ValueError(f"Unsupported teacher type: {teacher_type}")

    print("Teacher outputs generated:", len(teacher_outputs))

    return {
        **state,
        "teacher_outputs": teacher_outputs
    }

In [52]:
def student_training_agent(state: SLMGenState):

    plan = state["distillation_plan"]

    kd_strategy = plan["kd_strategy"]
    temperature = plan["temperature"]
    alpha = plan["alpha"]

    tokenized_dataset = state["tokenized_dataset"]
    student_model_name = state["student_model"]

    teacher_model = state["teacher_model_instance"]

    print("\n🎓 Student Training Agent")
    print("KD Strategy:", kd_strategy)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # --------------------------------------------------------
    # Detect architecture
    # --------------------------------------------------------

    config = AutoConfig.from_pretrained(student_model_name)

    if config.is_encoder_decoder:
        print("Student Model Type: Seq2Seq")
        student_model = AutoModelForSeq2SeqLM.from_pretrained(student_model_name)
    else:
        print("Student Model Type: Causal LM")
        student_model = AutoModelForCausalLM.from_pretrained(
            student_model_name,
            trust_remote_code=True,
            attn_implementation="eager"
        )

    student_model = student_model.to(device)

    student_model.train()

    # Teacher eval mode
    if state["teacher_type"] == "hf":
        teacher_model.eval()

    optimizer = AdamW(student_model.parameters(), lr=1e-5)

    tokenized_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    dataloader = DataLoader(
        tokenized_dataset["train"],
        batch_size=4,
        shuffle=False
    )

    epochs = 5

    # ========================================================
    # LOGIT KD TRAINING
    # ========================================================

    '''if kd_strategy in ["logit_kd", "hybrid_kd"]:

        print("Training using LOGIT KD")

        for epoch in range(epochs):

            total_loss = 0

            for batch in dataloader:

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                if "labels" in batch:
                    labels = batch["labels"].to(device)
                else:
                    labels = input_ids.clone()

                # mask padding tokens
                pad_token_id = state["student_tokenizer"].pad_token_id
                labels[labels == pad_token_id] = -100

                optimizer.zero_grad()

                # ----------------------------------------
                # Teacher forward (architecture aware)
                # ----------------------------------------

                with torch.no_grad():

                    if config.is_encoder_decoder:

                        decoder_input_ids = teacher_model._shift_right(labels)

                        teacher_outputs = teacher_model(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            decoder_input_ids=decoder_input_ids
                        )

                    else:

                        teacher_outputs = teacher_model(
                            input_ids=input_ids,
                            attention_mask=attention_mask
                        )

                    teacher_logits = teacher_outputs.logits

                    teacher_logits = torch.nan_to_num(teacher_logits, nan=0.0)
                    teacher_logits = torch.clamp(teacher_logits, -50, 50)
                    

                # ----------------------------------------
                # Student forward
                # ----------------------------------------

                if config.is_encoder_decoder:

                    student_outputs = student_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )

                else:

                    student_outputs = student_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=input_ids
                    )

                student_logits = student_outputs.logits
                student_logits = torch.nan_to_num(student_logits, nan=0.0)

                # align sequence length
                min_len = min(student_logits.size(1), teacher_logits.size(1))

                student_logits = student_logits[:, :min_len, :]
                teacher_logits = teacher_logits[:, :min_len, :]

                # clamp logits (prevents overflow)
                teacher_logits = torch.clamp(teacher_logits, -50, 50)

                # ----------------------------------------
                # KD Loss
                # ----------------------------------------

                student_log_probs = F.log_softmax(
                student_logits / temperature,
                dim=-1
                )

                teacher_probs = F.softmax(
                teacher_logits / temperature,
                dim=-1)

                # mask padding tokens
                mask = labels != -100
                mask = mask.unsqueeze(-1).expand_as(student_log_probs)

                student_log_probs = student_log_probs[mask]
                teacher_probs = teacher_probs[mask]

                kd_loss = F.kl_div(
                    student_log_probs,
                    teacher_probs,
                    reduction="batchmean"
                ) * (temperature ** 2)

                ce_loss = student_outputs.loss

                loss = alpha * kd_loss + (1 - alpha) * ce_loss

                if torch.isnan(loss):
                    print("⚠️ NaN detected, skipping batch")
                    continue

                loss.backward()

                torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)

                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(dataloader)

            print(f"Epoch {epoch+1} Loss:", total_loss)
            print(f"Epoch {epoch+1} Avg Loss:", avg_loss)'''
    


# ========================================================
# LOGIT KD TRAINING
# ========================================================

    if kd_strategy in ["logit_kd", "hybrid_kd"]:

        print("Training using LOGIT KD")

        for epoch in range(epochs):

            total_loss = 0
            steps = 0

            for batch in dataloader:

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                # mask padding tokens
                pad_token_id = state["student_tokenizer"].pad_token_id
                labels[labels == pad_token_id] = -100

                optimizer.zero_grad()

                # ----------------------------------------
                # Teacher forward
                # ----------------------------------------

                with torch.no_grad():

                    teacher_outputs = teacher_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )

                    teacher_logits = teacher_outputs.logits

                    teacher_logits = torch.nan_to_num(teacher_logits)
                    teacher_logits = torch.clamp(teacher_logits, -50, 50)

                # ----------------------------------------
                # Student forward
                # ----------------------------------------

                student_outputs = student_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                student_logits = student_outputs.logits
                student_logits = torch.nan_to_num(student_logits)

                # ----------------------------------------
                # Align sequence length
                # ----------------------------------------

                min_seq_len = min(
                    teacher_logits.size(1),
                    student_logits.size(1)
                )

                teacher_logits = teacher_logits[:, :min_seq_len, :]
                student_logits = student_logits[:, :min_seq_len, :]
                labels = labels[:, :min_seq_len]

                # ----------------------------------------
                # Align vocab size
                # ----------------------------------------

                min_vocab = min(
                    teacher_logits.size(-1),
                    student_logits.size(-1)
                )

                teacher_logits = teacher_logits[:, :, :min_vocab]
                student_logits = student_logits[:, :, :min_vocab]

                # ----------------------------------------
                # KD Loss
                # ----------------------------------------

                student_log_probs = F.log_softmax(
                    student_logits / temperature,
                    dim=-1
                )

                teacher_probs = F.softmax(
                    teacher_logits / temperature,
                    dim=-1
                )

                kd_loss = F.kl_div(
                    student_log_probs,
                    teacher_probs,
                    reduction="batchmean"
                ) * (temperature ** 2)

                ce_loss = student_outputs.loss

                loss = alpha * kd_loss + (1 - alpha) * ce_loss

                if torch.isnan(loss):
                    print("⚠️ NaN detected, skipping batch")
                    continue

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    student_model.parameters(),
                    1.0
                )

                optimizer.step()
                steps += 1

                total_loss += loss.item()

            avg_loss = total_loss / steps

            print(f"Epoch {epoch+1} Loss:", total_loss)
            print(f"Epoch {epoch+1} Avg Loss:", avg_loss)

    # ========================================================
    # SEQUENCE KD TRAINING
    # ========================================================

    elif kd_strategy == "sequence_kd":

        print("Training using SEQUENCE KD")

        teacher_outputs = state["teacher_outputs"]
        tokenizer = state["student_tokenizer"]

        teacher_tokens = tokenizer(
            teacher_outputs,
            padding="max_length",
            truncation=True,
            max_length=plan["max_output_length"],
            return_tensors="pt"
        )

        teacher_labels = teacher_tokens["input_ids"]

        teacher_labels[teacher_labels == tokenizer.pad_token_id] = -100

        dataset = tokenized_dataset["train"]

        for epoch in range(epochs):

            total_loss = 0

            for i, batch in enumerate(dataloader):

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                labels = teacher_labels[
                    i * dataloader.batch_size :
                    i * dataloader.batch_size + input_ids.size(0)
                ].to(device)

                optimizer.zero_grad()

                outputs = student_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss

                if torch.isnan(loss):
                    print("⚠️ NaN detected, skipping batch")
                    continue

                loss.backward()

                torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)

                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(dataloader)

            print(f"Epoch {epoch+1} Loss:", total_loss)
            print(f"Epoch {epoch+1} Avg Loss:", avg_loss)

    else:
        raise ValueError("Unsupported KD strategy")

    print("Student training completed")

    save_path = state.get("save_path", "./student_model")

    os.makedirs(save_path, exist_ok=True)

    print(f"\n💾 Saving student model to {save_path}")

    student_model.save_pretrained(save_path)

    tokenizer = state["student_tokenizer"]
    tokenizer.save_pretrained(save_path)

    print("✅ Model and tokenizer saved successfully")

    return {
        **state,
        "student_model_instance": student_model
    }


In [53]:
def student_inference_agent(state: SLMGenState):

    model = state["student_model_instance"]
    tokenizer = state["student_tokenizer"]
    dataset = state["processed_dataset"]

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model.eval()

    # choose evaluation split
    if "validation" in dataset:
        eval_split = dataset["validation"]
    else:
        eval_split = dataset["train"]

    predictions = []

    print("\n🤖 Generating student outputs...")

    for example in eval_split:

        inputs = tokenizer(
            example["input_text"],
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(device)

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_length=128
            )

        text = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        predictions.append(text)

    print("Student inference completed")

    return {
        **state,
        "student_outputs": predictions
    }

In [54]:


evaluation_planner_llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

EVAL_PLANNER_PROMPT = PromptTemplate(
    input_variables=["task"],
    template="""
You are an NLP evaluation planner.

Task: {task}

Choose the correct evaluation metric.

Rules:

summarization → rouge  
translation → bleu  
text_classification → accuracy  
nli → accuracy  
paraphrase → accuracy  
question_answering → squad  
instruction_tuning → rouge

Return JSON only:

{{
 "metric": "...",
 "mode": "text_generation | classification | qa"
}}
"""
)

def evaluation_planner_agent(state: SLMGenState):

    chain = (
        EVAL_PLANNER_PROMPT
        | evaluation_planner_llm
        | StrOutputParser()
    )

    response = chain.invoke({
        "task": state["task"]
    })

    plan = json.loads(response)

    print("\n📊 Evaluation Plan")
    print(plan)

    return {
        **state,
        "evaluation_plan": plan
    }

def evaluation_tool_agent(state: SLMGenState):

    plan = state["evaluation_plan"]

    metric_name = plan["metric"]

    dataset = state["processed_dataset"]
    predictions = state["student_outputs"]

    print("\n⚙️ Evaluation Tool Running")
    print("Metric:", metric_name)

    metric = evaluate.load(metric_name)

    # Use validation if available, otherwise fallback to train
    if "validation" in dataset:
        eval_split = dataset["validation"]
    else:
        print("⚠️ No validation split found. Using train split for evaluation.")
        eval_split = dataset["train"]

    references = [
        example["target_text"]
        for example in eval_split
    ]

    if metric_name == "rouge":

        results = metric.compute(
            predictions=predictions,
            references=references
        )

    elif metric_name == "bleu":

        results = metric.compute(
            predictions=predictions,
            references=[[r] for r in references]
        )

    elif metric_name == "accuracy":

        results = metric.compute(
            predictions=predictions,
            references=references
        )

    elif metric_name == "squad":

        predictions_formatted = [
            {"id": str(i), "prediction_text": p}
            for i, p in enumerate(predictions)
        ]

        references_formatted = [
            {"id": str(i), "answers": {"text": [r], "answer_start": [0]}}
            for i, r in enumerate(references)
        ]

        results = metric.compute(
            predictions=predictions_formatted,
            references=references_formatted
        )

    else:
        raise ValueError("Unsupported metric")

    print("\n📈 Evaluation Results")
    print(results)

    return {
        **state,
        "evaluation_metrics": results
    }



In [55]:

initial_state: SLMGenState = {

    "task": "question_answering",
    "user_quary": "i need to make a qustion answering dataset",
    "dataset_name": None,
    "dataset": None,
    "dataset_config": None,

    "detected_input_field": None,
    "detected_target_field": None,

    "distillation_plan": {},

    "clean_dataset": None,
    "processed_dataset": None,

    "tokenized_dataset": None,
    "student_tokenizer": None,

    "teacher_type": "openai",
    "teacher_model": "gpt-4o-mini",
    "teacher_model_instance": None,
    "teacher_tokenizer": None,

    "teacher_logits": None,
    "teacher_outputs": [],

    "student_model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "student_model_instance": None,

    "evaluation_metrics": {},
    "student_outputs": [],

    "save_path": "./student_model_QA_openai-phi3"
}


In [56]:

graph = StateGraph(SLMGenState)

graph.add_node("dataset_selector", dataset_selector_agent)
graph.add_node("dataset_loader", dataset_loader_agent)
graph.add_node("field_detection", llm_field_detection_agent)
graph.add_node("preprocessing", preprocessing_agent)
graph.add_node("planner", distillation_planner_agent)
graph.add_node("tokenizer", tokenizer_agent)
graph.add_node("teacher", teacher_agent)
graph.add_node("teacher_generation", teacher_generation_agent)

graph.add_node("student_training", student_training_agent)
graph.add_node("student_inference", student_inference_agent)
graph.add_node("evaluation_planner", evaluation_planner_agent)
graph.add_node("evaluation_tool", evaluation_tool_agent)



graph.set_entry_point("dataset_selector")


graph.add_edge("dataset_selector", "dataset_loader")
graph.add_edge("dataset_loader", "field_detection")
graph.add_edge("field_detection", "preprocessing")
graph.add_edge("preprocessing", "planner")
graph.add_edge("planner", "tokenizer")
graph.add_edge("tokenizer", "teacher")
graph.add_edge("teacher","teacher_generation")
graph.add_edge("teacher_generation", "student_training")
graph.add_edge("student_training", "student_inference")
graph.add_edge("student_inference", "evaluation_planner")
graph.add_edge("evaluation_planner", "evaluation_tool")
graph.add_edge("evaluation_tool", END)



app = graph.compile()



In [57]:
'''from langgraph.graph import StateGraph, END

graph = StateGraph(SLMGenState)

# =========================================================
# Nodes
# =========================================================

graph.add_node("dataset_selector", dataset_selector_agent)
graph.add_node("dataset_loader", dataset_loader_agent)
graph.add_node("field_detection", llm_field_detection_agent)
graph.add_node("preprocessing", preprocessing_agent)

graph.add_node("planner", distillation_planner_agent)
graph.add_node("tokenizer", tokenizer_agent)

graph.add_node("teacher", teacher_agent)
graph.add_node("teacher_generation", teacher_generation_agent)

graph.add_node("student_training", student_training_agent)
graph.add_node("student_inference", student_inference_agent)

graph.add_node("evaluation_planner", evaluation_planner_agent)
graph.add_node("evaluation_tool", evaluation_tool_agent)

# =========================================================
# Entry Point
# =========================================================

graph.set_entry_point("dataset_selector")

# =========================================================
# Edges
# =========================================================

graph.add_edge("dataset_selector", "dataset_loader")
graph.add_edge("dataset_loader", "field_detection")
graph.add_edge("field_detection", "preprocessing")

graph.add_edge("preprocessing", "planner")
graph.add_edge("planner", "tokenizer")

graph.add_edge("tokenizer", "teacher")

# Teacher model loading
graph.add_edge("teacher", "teacher_generation")

# Teacher generates logits or sequences
graph.add_edge("teacher_generation", "student_training")

# Train student
graph.add_edge("student_training", "student_inference")

# Evaluate
graph.add_edge("student_inference", "evaluation_planner")
graph.add_edge("evaluation_planner", "evaluation_tool")

graph.add_edge("evaluation_tool", END)

# =========================================================
# Compile Graph
# =========================================================

app = graph.compile()'''

'from langgraph.graph import StateGraph, END\n\ngraph = StateGraph(SLMGenState)\n\n# =========================================================\n# Nodes\n# =========================================================\n\ngraph.add_node("dataset_selector", dataset_selector_agent)\ngraph.add_node("dataset_loader", dataset_loader_agent)\ngraph.add_node("field_detection", llm_field_detection_agent)\ngraph.add_node("preprocessing", preprocessing_agent)\n\ngraph.add_node("planner", distillation_planner_agent)\ngraph.add_node("tokenizer", tokenizer_agent)\n\ngraph.add_node("teacher", teacher_agent)\ngraph.add_node("teacher_generation", teacher_generation_agent)\n\ngraph.add_node("student_training", student_training_agent)\ngraph.add_node("student_inference", student_inference_agent)\n\ngraph.add_node("evaluation_planner", evaluation_planner_agent)\ngraph.add_node("evaluation_tool", evaluation_tool_agent)\n\n# =========================================================\n# Entry Point\n# =============

In [58]:
final_state = app.invoke(initial_state)#

✅ Selected from registry: squad_v2
📦 Loading dataset: squad_v2
✅ Loaded from registry configuration
📊 Dataset limited to 1000 train samples
🔍 LLM Detected:
Input: question
Target: answers
🧹 Preprocessing for task: question_answering
Input field: question
Target field: answers
✅ Preprocessing completed successfully

🧠 Distillation Plan
{'kd_strategy': 'sequence_kd', 'requires_teacher_logits': False, 'tokenization': 'student_only', 'loss_function': 'sequence_cross_entropy', 'temperature': 1.0, 'alpha': 0.5, 'max_input_length': 512, 'max_output_length': 128}

🔤 Tokenizer Agent
KD Strategy: sequence_kd


c:\Users\i9\anaconda3\envs\slm_gen\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\i9\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 20/20 [00:00<00:00, 1054.28 examples/s]


✅ Tokenization complete

🧠 Teacher Agent
Teacher type: openai
Teacher model: gpt-4o-mini
Task: question_answering

🧠 Generating teacher outputs


100%|██████████| 200/200 [03:40<00:00,  1.10s/it]


Teacher outputs generated: 200

🎓 Student Training Agent
KD Strategy: sequence_kd
Student Model Type: Causal LM


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11278.78it/s]


Training using SEQUENCE KD


ValueError: Expected input batch_size (2048) to match target batch_size (512).

In [18]:
final_state

NameError: name 'final_state' is not defined

In [ ]:
datasetname = dataset_selector_agent(initial_state)
datasetloder = dataset_loader_agent(datasetname)


📦 Loading dataset: cnn_dailymail
✅ Loaded from registry configuration
📊 Dataset limited to 200 train samples


In [ ]:
llmfeild = llm_field_detection_agent(datasetloder)

🔍 LLM Detected:
Input: article
Target: highlights


In [ ]:
preprocced = preprocessing_agent(llmfeild)

🧹 Preprocessing for task: summarization
Input field: article
Target field: highlights
✅ Preprocessing completed successfully


In [ ]:
planner = distillation_planner_agent(preprocced)


🧠 Distillation Plan
{'kd_strategy': 'logit_kd', 'requires_teacher_logits': True, 'tokenization': 'shared', 'loss_function': 'kl_divergence', 'temperature': 2.0, 'alpha': 0.5, 'max_input_length': 512, 'max_output_length': 128}


In [ ]:
token = tokenizer_agent(planner)


🔤 Tokenizer Agent
KD Strategy: logit_kd
✅ Tokenization complete


In [ ]:
tech = teacher_agent(token)


🧠 Teacher Agent
Teacher type: hf
Teacher model: google-t5/t5-base


In [ ]:
tech_gen = teacher_generation_agent(tech)


🧠 Generating teacher outputs


100%|██████████| 50/50 [05:21<00:00,  6.44s/it]

✅ Teacher outputs generated


In [ ]:
stu = student_training_agent(tech_gen)


🎓 Student Training Agent
KD Strategy: logit_kd
Student Model Type: Seq2Seq
Training using LOGIT KD
⚠️ NaN detected, skipping batch
⚠️ NaN detected, skipping batch
⚠️ NaN detected, skipping batch
⚠️ NaN detected, skipping batch
⚠️ NaN detected, skipping batch
⚠️ NaN detected, skipping batch
⚠️ NaN detected, skipping batch


KeyboardInterrupt: 

In [ ]:
tech_gen

{'task': 'summarization',
 'user_quary': 'make me a text summerisation system',
 'dataset_name': 'cnn_dailymail',
 'dataset': DatasetDict({
     train: Dataset({
         features: ['article', 'highlights', 'id'],
         num_rows: 200
     })
     validation: Dataset({
         features: ['article', 'highlights', 'id'],
         num_rows: 20
     })
     test: Dataset({
         features: ['article', 'highlights', 'id'],
         num_rows: 20
     })
 }),
 'dataset_config': {'name': 'cnn_dailymail',
  'hf_id': 'cnn_dailymail',
  'config': '3.0.0',
  'input_field': 'article',
  'target_field': 'highlights',
  'metric': 'rouge'},
 'detected_input_field': 'article',
 'detected_target_field': 'highlights',
 'distillation_plan': {'kd_strategy': 'logit_kd',
  'requires_teacher_logits': True,
  'tokenization': 'shared',
  'loss_function': 'kl_divergence',
  'temperature': 2.0,
  'alpha': 0.5,
  'max_input_length': 512,
  'max_output_length': 128},
 'clean_dataset': None,
 'processed_dataset

In [ ]:
len(tech_gen["teacher_outputs"])

200

In [ ]:
print(tech_gen["processed_dataset"]["train"][0]["input_text"])
print(tech_gen["teacher_outputs"][0])

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Details of how